In [0]:
# ============================================================
# Notebook: 02_silver_online_retail_transform
# Purpose: Clean and standardise and validate Bronze transactions 
#          into Silver
# Author: Dele Fatoba
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_timestamp,
    regexp_replace,
    current_timestamp,
    when
)

In [0]:
bronze_table_name = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "bronze."
    "online_retail_transactions"
)

silver_table_name = (
    "retaildataops_dbw_dev_v4ptce_7405619702729069."
    "silver."
    "online_retail_transactions"
)


In [0]:
bronze_df = spark.table(bronze_table_name)

display(bronze_df.limit(20))
bronze_df.printSchema()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,ingestion_timestamp,source_file
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv


root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed("Invoice", "invoice_no")
    .withColumnRenamed("StockCode", "stock_code")
    .withColumnRenamed("Description", "product_description")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("InvoiceDate", "invoice_date")
    .withColumnRenamed("Price", "unit_price")
    .withColumnRenamed("Customer ID", "customer_id")
    .withColumnRenamed("Country", "country")
)

In [0]:
silver_df = (
    silver_df
    .withColumn("invoice_no", trim(col("invoice_no").cast("string")))
    .withColumn("stock_code", trim(col("stock_code").cast("string")))
    .withColumn("product_description", trim(col("product_description")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("invoice_date", to_timestamp(col("invoice_date")))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("customer_id", regexp_replace(col("customer_id").cast("string"), "\\.0$", ""))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn(
        "transaction_type",
        when(col("quantity") < 0, "RETURN").otherwise("SALE")
    )
    .withColumn("sales_amount", col("quantity") * col("unit_price"))
    .withColumn("silver_processed_timestamp", current_timestamp())
)

In [0]:
silver_df = (
    silver_df
    .filter(col("invoice_no").isNotNull())
    .filter(col("stock_code").isNotNull())
    .filter(col("invoice_date").isNotNull())
    .filter(col("unit_price") > 0)
    .dropDuplicates(["invoice_no", "stock_code", "invoice_date", "customer_id"])
)

In [0]:
display(silver_df.limit(20))
silver_df.printSchema()

print(f"Silver record count: {silver_df.count():,}")

invoice_no,stock_code,product_description,quantity,invoice_date,unit_price,customer_id,country,ingestion_timestamp,source_file,transaction_type,sales_amount,silver_processed_timestamp
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,83.4,2026-06-02T14:26:18.396Z
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,81.0,2026-06-02T14:26:18.396Z
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,81.0,2026-06-02T14:26:18.396Z
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,100.80000000000001,2026-06-02T14:26:18.396Z
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,30.0,2026-06-02T14:26:18.396Z
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,39.599999999999994,2026-06-02T14:26:18.396Z
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,30.0,2026-06-02T14:26:18.396Z
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,59.5,2026-06-02T14:26:18.396Z
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,30.599999999999998,2026-06-02T14:26:18.396Z
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,45.0,2026-06-02T14:26:18.396Z


root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- transaction_type: string (nullable = false)
 |-- sales_amount: double (nullable = true)
 |-- silver_processed_timestamp: timestamp (nullable = false)

Silver record count: 1,015,468


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retaildataops_dbw_dev_v4ptce_7405619702729069.silver
""")

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)

print(f"Silver Delta table created: {silver_table_name}")


Silver Delta table created: retaildataops_dbw_dev_v4ptce_7405619702729069.silver.online_retail_transactions


In [0]:
display(
    spark.table(silver_table_name)
    .limit(20)
)

invoice_no,stock_code,product_description,quantity,invoice_date,unit_price,customer_id,country,ingestion_timestamp,source_file,transaction_type,sales_amount,silver_processed_timestamp
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,83.4,2026-06-02T14:28:06.788Z
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,81.0,2026-06-02T14:28:06.788Z
489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01T09:06:00.000Z,1.69,13078,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,27.04,2026-06-02T14:28:06.788Z
489437,21351,CINAMMON & ORANGE WREATH,2,2009-12-01T09:08:00.000Z,6.75,15362,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,13.5,2026-06-02T14:28:06.788Z
489437,21987,PACK OF 6 SKULL PAPER CUPS,12,2009-12-01T09:08:00.000Z,0.65,15362,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,7.800000000000001,2026-06-02T14:28:06.788Z
489438,84031A,CHARLIE+LOLA RED HOT WATER BOTTLE,56,2009-12-01T09:24:00.000Z,3.0,18102,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,168.0,2026-06-02T14:28:06.788Z
489439,22333,RETRO SPORT PARTY BAG + STICKER SET,8,2009-12-01T09:28:00.000Z,1.65,12682,FRANCE,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,13.2,2026-06-02T14:28:06.788Z
489441,84029E,RED WOOLLY HOTTIE WHITE HEART.,36,2009-12-01T09:44:00.000Z,2.95,18087,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,106.2,2026-06-02T14:28:06.788Z
489442,22111,SCOTTIE DOG HOT WATER BOTTLE,3,2009-12-01T09:46:00.000Z,4.95,13635,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,14.850000000000001,2026-06-02T14:28:06.788Z
489442,84899E,YELLOW + BROWN BEAR FELT PURSE KIT,12,2009-12-01T09:46:00.000Z,1.25,13635,UNITED KINGDOM,2026-06-02T14:20:20.742Z,dbfs:/Volumes/retaildataops_dbw_dev_v4ptce_7405619702729069/raw/online_retail_volume/online_retail_II.csv,SALE,15.0,2026-06-02T14:28:06.788Z
